In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

!mkdir -p /content/work
!cp "/content/drive/MyDrive/colab_upload/train.py" /content/work/
!unzip -q "/content/drive/MyDrive/colab_upload/dataset.zip" -d /content/work/


Mounted at /content/drive


In [2]:
!cp "/content/drive/MyDrive/colab_upload/train.py" /content/work/

In [3]:
!pip install -q transformers datasets evaluate accelerate jiwer soundfile librosa peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 127.0 MB/s eta 0:00:00


In [4]:
!pip install -q "transformers==4.46.3" "accelerate==1.1.1"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 138.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.2/333.2 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 121.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [5]:
!ls /content/work/dataset
!ls /content/work/dataset/train | head


test  train  val
audio
metadata.jsonl
metadata.jsonl.bak


In [7]:
import warnings
warnings.filterwarnings("ignore", message=".*Trainer.tokenizer is now deprecated.*")

%cd /content/work
!python train.py --model openai/whisper-small --dataset ./dataset --output ./model_v3 --backup-dir /content/drive/MyDrive/makeup-stt/checkpoints


/content/work
2026-07-14 18:46:00.337739: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-14 18:46:00.406325: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Whisper fine-tuning
Model:        openai/whisper-small
Dataset:      ./dataset
Output:       ./model_v3
Backup:       /content/drive/MyDrive/makeup-stt/checkpoints
Max steps:    4000
Batch size:   8
Grad accum:   2
Learning rate:5e-06
CUDA:         True
GPU:          NVIDIA L4
No local checkpoint — restoring /content/drive/MyDrive/makeup-stt/checkpoints/chec

In [12]:
import os
os.listdir("./dataset/test/audio")[:5]


['E5FiU4NuEtc_step6.wav',
 '15NMOAJBekg_step20.wav',
 '2gOtXiUzB58_step6.wav',
 'zH8hw1zpd-8_step10.wav',
 '71R-xgHVe9k_step3.wav']

In [13]:
from transformers import WhisperForConditionalGeneration, WhisperProcessor
import torch, librosa

model = WhisperForConditionalGeneration.from_pretrained('./model_v3/final')
processor = WhisperProcessor.from_pretrained('./model_v3/final')

audio, sr = librosa.load("./dataset/test/audio/E5FiU4NuEtc_step6.wav", sr=16000)
inputs = processor(audio, sampling_rate=16000, return_tensors="pt").input_features

with torch.no_grad():
    predicted_ids = model.generate(inputs, max_length=225)

transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
print(transcription)


You have passed task=transcribe, but also have set `forced_decoder_ids` to [[1, 50259], [2, 50359], [3, 50363]] which creates a conflict. `forced_decoder_ids` will be ignored in favor of task=transcribe.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


I'm going to apply this matte eyeshadow all over my eyelid and it's a beautiful nude sand tone which is perfect for the color mood of this look and also really shape the eye but in a very natural way because then I'm going to use this crease eyeshadow that is sort of a boire de rose and really shape my crease and the outside corner in a more pointy way to really accentuate this cat eye effect.


In [14]:
!cp "/content/drive/MyDrive/colab_upload/evaluate_test.py" /content/work/

In [16]:
!python evaluate_test.py --model ./model_v3/final --dataset ./dataset

Loading model from ./model_v3/final on cuda...
Loading test split...
Resolving data files: 100% 5386/5386 [00:00<00:00, 55716.47it/s]
Resolving data files: 100% 1240/1240 [00:00<00:00, 32109.90it/s]
Resolving data files: 100% 1005/1005 [00:00<00:00, 29037.02it/s]
You have passed task=transcribe, but also have set `forced_decoder_ids` to [[1, 50259], [2, 50359], [3, 50363]] which creates a conflict. `forced_decoder_ids` will be ignored in favor of task=transcribe.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
8/1003
16/1003
24/1003
32/1003
40/1003
48/1003
56/1003
64/1003
72/1003
80/1003
88/1003
96/1003
104/1003
112/1003
120/1003
128/1003
136/1003
144/1003
152/1003
160/1003
168/1003
176/1003
184/1003
192/1003
200/1003
208/1003
216/1003
224/1003
232/1003
240/1003
248/1003
256/1003
264/1003
272/1003
280/1003
2

In [17]:
import shutil
shutil.copytree("./model_v3/final", "/content/drive/MyDrive/makeup-stt/model_final", dirs_exist_ok=True)


'/content/drive/MyDrive/makeup-stt/model_final'